# VisionVoice Modelling Notebook

This notebook is the individual modelling submission. It reuses the shared data-preparation workflow, trains **Model 1** (baseline ResNet-LSTM), then trains **Model 2** (ResNet-LSTM with attention) based on the Phase 2 findings. Both architectures are evaluated with BLEU metrics and qualitative visual inspection.


# 1. Environment and Configuration


## 1.1 Runtime Check


In [ ]:
import os
import platform
import sys

print(f'Python : {sys.version}')
print(f'OS     : {platform.system()} {platform.release()}')
print(f'CWD    : {os.getcwd()}')

## 1.2 Libraries and Device Setup

This section imports the scientific Python, PyTorch, image-processing, NLP, and visualization utilities used throughout the notebook. It also selects the best available compute device.


In [ ]:
# Standard Python Utilities
import os, sys, gc, time, json, random, zipfile, warnings, shutil
from pathlib import Path
from typing import List, Dict, Tuple, Optional, Iterable, Any
from collections import Counter

# Networking
import requests

# Data Processing
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

# Image Processing
from PIL import Image, UnidentifiedImageError

# Visualization
import matplotlib.pyplot as plt

# Deep Learning (PyTorch)
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
import torchvision.models as models
import torchvision.transforms as transforms

# NLP & Progress
import nltk
from nltk.translate.bleu_score import corpus_bleu
from tqdm import tqdm

# Suppress non-critical warnings for cleaner output
warnings.filterwarnings('ignore')

# Centralized Device Configuration
# This ensures we have a single source of truth for hardware acceleration
DEVICE = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)

print(f"Python       : {sys.version.split()[0]}")
print(f"PyTorch      : {torch.__version__}")
print(f"Target Device: {DEVICE}")

## 1.3 Configuration

The `Settings` class centralizes paths, runtime toggles, split proportions, model hyperparameters, and evaluation settings so experiments are reproducible across Kaggle and local environments.


In [ ]:
class Settings:
    """Centralized configuration class for the Image Captioning project.

    Contains all hyperparameters, environment settings, and dynamic path
    routing for both Kaggle and local VS Code environments.
    """

    # 1. Operational Toggles
    MODE: str = "train"   # "eda", "train", or "inference"
    DEBUG: bool = False
    DEBUG_SAMPLE_SIZE: int = 50 # Number of files to audit when DEBUG is True
    SHOW_EDA_PLOTS: bool = MODE == "eda"  # Dataset/EDA figures
    SHOW_MODEL_PLOTS: bool = MODE in {"train", "inference"}  # Post-training diagnostics

    # 2. Environment Detection
    IS_KAGGLE: bool = Path("/kaggle/input").exists()

    # 3. Dynamic Path Management
    # Override these in Kaggle with environment variables if your dataset slug changes:
    # VIZWIZ_DATA_DIR, VIZWIZ_ANNOTATIONS_DIR, VIZWIZ_IMAGE_DIR
    if IS_KAGGLE:
        DATA_DIR = Path(os.getenv("VIZWIZ_DATA_DIR", "/kaggle/input/datasets/tuannm3823/vizwiz"))
        ANNOTATIONS_DIR = Path(os.getenv("VIZWIZ_ANNOTATIONS_DIR", str(DATA_DIR / "annotations" / "annotations")))
        TRAIN_IMG_DIR = Path(os.getenv("VIZWIZ_IMAGE_DIR", str(DATA_DIR / "val" / "val")))
        VAL_IMG_DIR = TRAIN_IMG_DIR
        WORK_DIR = Path("/kaggle/working")
    else:
        try:
            from google.colab import drive
            drive.mount('/content/drive')
            DATA_DIR = Path(os.getenv("VIZWIZ_LOCAL_DATA_DIR", "/content/drive/MyDrive/Sem3_2026 Autumn/94691 Deep Learning/dl_assignments/dl_at3"))
        except ImportError:
            DATA_DIR = Path(os.getenv("VIZWIZ_LOCAL_DATA_DIR", "./data"))

        ANNOTATIONS_DIR = DATA_DIR / "annotations"
        TRAIN_IMG_DIR = DATA_DIR / "images"
        VAL_IMG_DIR = DATA_DIR / "images"
        WORK_DIR = DATA_DIR / "working"

    BASE: Path = DATA_DIR
    CACHE_DIR: Path = WORK_DIR / "cache"
    WORK_CACHE_DIR: Path = CACHE_DIR / "work"

    # /kaggle/input/models/tuannm3823/visionvoice-image-captioning/pytorch/attention-resnet-lstm/1/vision_voice_attention_best.pth
    # /kaggle/input/models/tuannm3823/visionvoice-image-captioning/pytorch/baseline-resnet-lstm/1/vision_voice_baseline_best.pth

    # 4. Remote Data URLs (For Section 3.1)
    URL_ANNOTATIONS: str = "https://vizwiz.cs.colorado.edu/VizWiz_final/annotations/val.json"
    URL_IMAGES: str = "https://vizwiz.cs.colorado.edu/VizWiz_final/images/val.zip"

    # 5. Data Processing & Pipeline Parameters
    SEED: int = 42
    VAL_SPLIT_SIZE: float = 0.1    # 10% internal validation split for model selection
    TEST_SPLIT_SIZE: float = 0.1   # 10% internal test split for final reporting
    VOCAB_MIN_FREQ: int = 2        # Minimum word occurrences to be indexed
    IMAGE_SIZE: int = 224          # ResNet standard input size
    NUM_WORKERS: int = 2           # DataLoader subprocesses

    # 6. Model Hyperparameters
    LEARNING_RATE: float = 1e-4
    BATCH_SIZE: int = 32
    EPOCHS: int = 10
    MAX_SEQ_LEN: int = 25          # Max words generated during inference
    EMBED_SIZE: int = 256          # Dimension of shared visual/textual space
    HIDDEN_SIZE: int = 512         # Number of units in LSTM hidden state
    NUM_LAYERS: int = 1            # Number of LSTM layers stacked
    MAX_GRAD_NORM: float = 5.0     # Clipping threshold for vanishing/exploding gradients

    # 7. Evaluation Parameters
    BLEU_EVAL_SAMPLES: int = 500   # Number of validation images to test for BLEU scoring
    DECODING_STRATEGY: str = "greedy"  # "greedy" or "beam"
    BEAM_SIZE: int = 3
    REPETITION_PENALTY: float = 1.2

# Instantiate configuration object
CFG = Settings()

# Post-instantiation directory setup
CFG.CACHE_EXISTS = CFG.CACHE_DIR.exists()
CFG.WORK_CACHE_DIR.mkdir(parents=True, exist_ok=True)

print(f'MODE         : {CFG.MODE}')
print(f'BASE         : {CFG.BASE}')
print(f'CACHE_EXISTS : {CFG.CACHE_EXISTS}')
print(f'CACHE_DIR    : {CFG.CACHE_DIR}')

## 1.4 Memory Utilities

These helpers clear accelerator memory and reduce DataFrame memory usage. They keep long Kaggle training sessions more stable.


In [ ]:
def clear_memory() -> None:
    """Forces garbage collection and clears VRAM cache to prevent OOM errors."""
    gc.collect()

    # Clear CUDA cache if using NVIDIA
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    # Clear MPS cache if using Apple Silicon
    elif torch.backends.mps.is_available():
        torch.mps.empty_cache()

    if CFG.DEBUG:
        print("[Memory] Garbage collection forced and accelerator cache cleared.")


def reduce_mem_usage(df: pd.DataFrame) -> pd.DataFrame:
    """
    Iterates through all columns of a dataframe and downcasts data types
    to their smallest possible representation to reduce memory footprint.
    """
    start_mem = df.memory_usage().sum() / 1024**2

    for col in df.columns:
        col_type = df[col].dtype

        # Exclude object (string) and categorical types from numeric downcasting
        if col_type != object and not pd.api.types.is_categorical_dtype(col_type):
            c_min = df[col].min()
            c_max = df[col].max()

            if str(col_type)[:3] == 'int':
                if c_min > np.iinfo(np.int8).min and c_max < np.iinfo(np.int8).max:
                    df[col] = df[col].astype(np.int8)
                elif c_min > np.iinfo(np.int16).min and c_max < np.iinfo(np.int16).max:
                    df[col] = df[col].astype(np.int16)
                elif c_min > np.iinfo(np.int32).min and c_max < np.iinfo(np.int32).max:
                    df[col] = df[col].astype(np.int32)
                else:
                    df[col] = df[col].astype(np.int64)
            else:
                if c_min > np.finfo(np.float16).min and c_max < np.finfo(np.float16).max:
                    df[col] = df[col].astype(np.float16)
                elif c_min > np.finfo(np.float32).min and c_max < np.finfo(np.float32).max:
                    df[col] = df[col].astype(np.float32)
                else:
                    df[col] = df[col].astype(np.float64)

    end_mem = df.memory_usage().sum() / 1024**2

    if CFG.DEBUG:
        reduction = 100 * (start_mem - end_mem) / start_mem
        print(f"[Memory] DataFrame compressed from {start_mem:.2f}MB to {end_mem:.2f}MB ({reduction:.1f}% reduction).")

    return df

## 1.5 Reproducibility

This cell fixes Python, NumPy, and PyTorch seeds so split generation and model training are as reproducible as possible.


In [ ]:
def seed_everything(seed: int = CFG.SEED) -> None:
    """
    Locks all random number generators to a specific seed to ensure
    Global Determinism across independent experimental runs.
    """
    # 1. Python and Data structures
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)

    # 2. NumPy
    np.random.seed(seed)

    # 3. PyTorch Core
    torch.manual_seed(seed)

    # 4. PyTorch CUDA (Hardware Acceleration)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed) # Ensures determinism for multi-GPU setups

        # Enforce deterministic convolutions.
        # Note: This may slightly reduce training speed, but guarantees 100% reproducible results.
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

    print(f"Global seed successfully locked to: {seed}")

# Execute initialization
seed_everything(CFG.SEED)

# 2. Shared Data Preparation


## 2.1 Dataset Loading

The assignment uses only the official VizWiz-Captions validation set. The notebook supports Kaggle-mounted data first and falls back to downloading the public validation annotations/images for local runs.


In [ ]:
def download_file(url: str, dest_dir: Path, filename: str = None) -> Path:
    """
    Downloads a file from a URL using chunked streaming to prevent OOM errors.

    Args:
        url (str): The source URL.
        dest_dir (Path): The local directory to save the file.
        filename (str, optional): Custom filename. Defaults to URL's trailing name.

    Returns:
        Path: The absolute path to the downloaded file.
    """
    dest_dir.mkdir(parents=True, exist_ok=True)
    filename = filename or url.split("/")[-1]
    file_path = dest_dir / filename

    if file_path.exists():
        print(f"[Network] '{filename}' already exists in cache. Skipping download.")
        return file_path

    print(f"[Network] Initiating stream for '{filename}'...")
    try:
        # Stream=True prevents loading the entire file into RAM at once
        with requests.get(url, stream=True, timeout=15) as response:
            response.raise_for_status()
            with open(file_path, "wb") as file:
                for chunk in response.iter_content(chunk_size=8192):
                    if chunk:
                        file.write(chunk)
        print(f"[Network] Successfully downloaded: {file_path}")
    except requests.exceptions.RequestException as e:
        print(f"[Error] Failed to download {filename}: {e}")
        raise SystemExit("Critical data missing. Pipeline halted.")

    return file_path

def prepare_dataset() -> None:
    """
    Orchestrates the data pipeline: routing Kaggle paths or downloading/extracting
    the VizWiz dataset into the local Colab environment.
    """
    if CFG.IS_KAGGLE:
        print("[Pipeline] Kaggle environment detected. Utilizing read-only mounted datasets.")
        if not (CFG.ANNOTATIONS_DIR / "val.json").exists():
             print(f"[Warning] val.json not found in {CFG.ANNOTATIONS_DIR}.")
        return

    print("--- Initializing Local Data Pipeline ---")

    # 1. Acquire Annotations (Metadata)
    CFG.ANNOTATIONS_DIR.mkdir(parents=True, exist_ok=True)
    download_file(
        url=CFG.URL_ANNOTATIONS,
        dest_dir=CFG.ANNOTATIONS_DIR,
        filename="val.json"
    )

    # 2. Acquire and Extract Images
    # We verify extraction success by checking if the directory exists and is populated
    if CFG.TRAIN_IMG_DIR.exists() and any(CFG.TRAIN_IMG_DIR.iterdir()):
        print(f"[Pipeline] Image payload verified at: {CFG.TRAIN_IMG_DIR}. Skipping extraction.")
    else:
        print(f"[Pipeline] Image directory empty. Beginning extraction protocol...")
        CFG.TRAIN_IMG_DIR.mkdir(parents=True, exist_ok=True)

        # Download the heavy zip file to the temporary cache
        zip_path = download_file(CFG.URL_IMAGES, CFG.WORK_CACHE_DIR)

        print(f"[Storage] Unpacking compressed images to {CFG.TRAIN_IMG_DIR}...")
        try:
            with zipfile.ZipFile(zip_path, 'r') as zip_ref:
                # Extract to a temp cache folder to handle unexpected internal nested folders
                temp_extract_path = CFG.WORK_CACHE_DIR / "temp_extract"
                temp_extract_path.mkdir(exist_ok=True)
                zip_ref.extractall(temp_extract_path)

                # Flatten the directory structure if the zip contained a root 'val' folder
                extracted_items = list(temp_extract_path.iterdir())
                if len(extracted_items) == 1 and extracted_items[0].is_dir():
                    source_dir = extracted_items[0]
                else:
                    source_dir = temp_extract_path

                # Move files to final persistent directory
                for item in source_dir.iterdir():
                    shutil.move(str(item), str(CFG.TRAIN_IMG_DIR / item.name))

                # Housekeeping: Clean up temporary extraction folders
                shutil.rmtree(temp_extract_path)

            print("[Storage] Extraction and cleanup complete.")

        except zipfile.BadZipFile:
            print(f"[Error] Corrupted archive detected at {zip_path}.")
            zip_path.unlink() # Delete corrupted zip to force fresh download on next run
            raise SystemExit("Please re-run the cell to attempt download again.")

# Execute Pipeline
prepare_dataset()

## 2.2 Annotation Parsing

VizWiz stores image metadata and captions in separate JSON arrays. This step merges them into a caption-level table keyed by `file_name`.


In [ ]:
print(f"--- Parsing Raw Annotations ---")
json_path = CFG.ANNOTATIONS_DIR / "val.json"

# 1. Load the Raw Data
with open(json_path, 'r', encoding='utf-8') as f:
    data = json.load(f)

images_df = pd.DataFrame(data['images'])
annotations_df = pd.DataFrame(data['annotations'])

# 2. Basic Text Cleanup (Fix legacy carriage returns that break tokenizers)
annotations_df['caption'] = annotations_df['caption'].str.replace('\r', '', regex=False)

# 3. Initial Merge (Attach file_names to captions for Visualization & EDA)
raw_corpus_df = pd.merge(
    annotations_df,
    images_df[['id', 'file_name']],
    left_on='image_id',
    right_on='id',
    how='inner'
)
# Rename the annotation's 'id' column to avoid confusion, drop the duplicate image 'id'
raw_corpus_df = raw_corpus_df.drop(columns=['id_y']).rename(columns={'id_x': 'annotation_id'})

print(f"Successfully loaded {len(raw_corpus_df)} raw annotations.\n")

## 2.3 Raw Sample Inspection

This optional EDA cell displays a small sample of raw images and captions, including rejected and precanned labels. It is disabled in training mode to keep modelling runs lightweight.


In [ ]:
def show_raw_sample(df: pd.DataFrame, image_dir: Path, num_samples: int = 1) -> None:
    """Displays compact static image-caption samples for raw data inspection."""
    print(f"--- [Visualization] Sampling {num_samples} raw images ---")
    grouped = df.groupby('file_name')
    unique_files = list(grouped.groups.keys())
    samples = random.sample(unique_files, k=min(num_samples, len(unique_files)))

    for file_name in samples:
        group = grouped.get_group(file_name)
        img_path = image_dir / file_name

        fig, axes = plt.subplots(1, 2, figsize=(9, 3.8), dpi=100, gridspec_kw={'width_ratios': [1, 1.35]})

        if img_path.exists():
            try:
                with Image.open(img_path) as img:
                    axes[0].imshow(img.convert("RGB"))
            except Exception as e:
                axes[0].text(0.5, 0.5, f"Could not render image\n{e}", ha="center", va="center", wrap=True)
        else:
            axes[0].text(0.5, 0.5, "Image missing", ha="center", va="center")

        axes[0].set_title(file_name, fontsize=9)
        axes[0].axis("off")
        axes[1].axis("off")

        y = 0.98
        for _, row in group.iterrows():
            is_rejected = bool(row.get('is_rejected', 0.0))
            is_precanned = bool(row.get('is_precanned', 0.0))
            prefix = "[REJECTED] " if is_rejected else "[PRECANNED] " if is_precanned else ""
            color = plt.colormaps['viridis'](0.95) if is_rejected else plt.colormaps['viridis'](0.55) if is_precanned else 'black'
            axes[1].text(0.0, y, prefix + str(row['caption']), fontsize=8.5, color=color, va="top", wrap=True)
            y -= 0.18

        plt.tight_layout()
        plt.show()
        plt.close(fig)

# Execute Visualization
if CFG.SHOW_EDA_PLOTS:
    # Using a seed before sampling ensures we get the same random images every time we run the cell
    random.seed(CFG.SEED)
    show_raw_sample(raw_corpus_df, CFG.TRAIN_IMG_DIR, num_samples=3)


## 2.4 Caption Quality Audit

Before filtering, we quantify rejected and precanned captions so the cleaning policy is transparent and justified.


In [ ]:
print("--- [EDA] Raw Dataset Quality Analysis ---")

# 1. Identify Quality Flags
# Safely handle JSON boolean/float parsing variations
is_p = (raw_corpus_df['is_precanned'] == True) | (raw_corpus_df['is_precanned'] == 1.0)
is_r = (raw_corpus_df['is_rejected'] == True) | (raw_corpus_df['is_rejected'] == 1.0)

# 2. Annotation-Level Intersection Analysis
both_count = (is_p & is_r).sum()
just_precanned = (is_p & ~is_r).sum()
just_rejected = (is_r & ~is_p).sum()
clean_count = (~is_p & ~is_r).sum()

print("\n[Annotation Breakdown]")
print(f"Total Annotations : {len(raw_corpus_df)}")
print(f"Clean/Descriptive : {clean_count}")
print(f"Precanned Only    : {just_precanned}")
print(f"Rejected Only     : {just_rejected}")
print(f"BOTH (P & R)      : {both_count}")

# 3. Image-Level "Good Caption" Distribution
raw_corpus_df['is_good'] = ~is_p & ~is_r
dist_counts = raw_corpus_df.groupby('file_name')['is_good'].sum().value_counts().sort_index()

if CFG.SHOW_EDA_PLOTS:
    fig, axes = plt.subplots(1, 2, figsize=(10, 4.2), dpi=100)
    cmap = plt.colormaps['viridis']
    labels = ['Clean', 'Precanned', 'Rejected', 'Both']
    values = [clean_count, just_precanned, just_rejected, both_count]
    colors = [cmap(x) for x in np.linspace(0.18, 0.88, len(labels))]

    def pct_label(pct: float) -> str:
        return f'{pct:.1f}%' if pct >= 4 else ''

    wedges, _, autotexts = axes[0].pie(
        values,
        labels=None,
        autopct=pct_label,
        startangle=90,
        counterclock=False,
        colors=colors,
        pctdistance=0.74,
        wedgeprops={'width': 0.42, 'edgecolor': 'white', 'linewidth': 1.0},
        textprops={'fontsize': 8, 'color': 'black'}
    )
    axes[0].set_title('Raw Annotation Quality', fontsize=10)
    axes[0].legend(
        wedges,
        [f'{label}: {value:,} ({value / len(raw_corpus_df):.1%})' for label, value in zip(labels, values)],
        loc='center left',
        bbox_to_anchor=(1.0, 0.5),
        frameon=False,
        fontsize=8
    )

    bar_colors = [cmap(x) for x in np.linspace(0.25, 0.85, len(dist_counts))]
    axes[1].bar(dist_counts.index.astype(str), dist_counts.values, color=bar_colors)
    axes[1].set_title('Descriptive Captions per Image', fontsize=10)
    axes[1].set_xlabel('Valid captions')
    axes[1].set_ylabel('Number of images')
    axes[1].grid(axis='y', alpha=0.25)

    plt.tight_layout()
    plt.show()
    plt.close(fig)

# Cleanup temporary EDA column
raw_corpus_df.drop(columns=['is_good'], inplace=True)


### Data Quality Takeaways

* The official VizWiz validation set contains substantial real-world noise.
* Rejected and precanned captions should not be used as target sentences.
* Cleaning is required before building the vocabulary or training models.


## 2.5 Cleaning and Internal Splitting

The source data is the official VizWiz validation set. Within that source set, we create internal image-level train/validation/test splits. Splitting by image prevents captions for the same image from leaking across model-selection and final-evaluation stages.


In [ ]:
print("--- [Data Filtering] ---")
original_anno_count = len(raw_corpus_df)

# 1. Silently drop Spam and Precanned
clean_corpus_df = raw_corpus_df[
    (raw_corpus_df['is_rejected'] != True) &
    (raw_corpus_df['is_rejected'] != 1.0) &
    (raw_corpus_df['is_precanned'] != True) &
    (raw_corpus_df['is_precanned'] != 1.0)
].reset_index(drop=True)

spam_removed = original_anno_count - len(clean_corpus_df)
print(f"Dropped {spam_removed} noisy annotations.")

# 2. Identify and drop orphaned images (images with 0 valid captions)
# By grouping by file_name, we ensure we only keep images that still exist in clean_corpus_df
valid_images = clean_corpus_df['file_name'].unique()
print(f"Final valid corpus size: {len(clean_corpus_df)} captions across {len(valid_images)} images.\n")

print("--- [Splitting] ---")
# 3. Perform leakage-free internal splits from the official VizWiz validation set.
# The assignment restricts source data to VizWiz val, but still asks for train/validation/test splits.
# We split by image file name so captions for the same image never leak across splits.
holdout_size = CFG.VAL_SPLIT_SIZE + CFG.TEST_SPLIT_SIZE
train_imgs, holdout_imgs = train_test_split(
    valid_images,
    test_size=holdout_size,
    random_state=CFG.SEED
)

relative_test_size = CFG.TEST_SPLIT_SIZE / holdout_size
val_imgs, test_imgs = train_test_split(
    holdout_imgs,
    test_size=relative_test_size,
    random_state=CFG.SEED
)

train_df = clean_corpus_df[clean_corpus_df['file_name'].isin(train_imgs)].reset_index(drop=True)
val_df = clean_corpus_df[clean_corpus_df['file_name'].isin(val_imgs)].reset_index(drop=True)
test_df = clean_corpus_df[clean_corpus_df['file_name'].isin(test_imgs)].reset_index(drop=True)

# Compress datatypes to save memory
train_df = reduce_mem_usage(train_df)
val_df = reduce_mem_usage(val_df)
test_df = reduce_mem_usage(test_df)

print(f"Train Split : {len(train_imgs)} images | {len(train_df)} total captions")
print(f"Val Split   : {len(val_imgs)} images | {len(val_df)} total captions")
print(f"Test Split  : {len(test_imgs)} images | {len(test_df)} total captions\n")

# 4. Handle Debug Truncation
if CFG.DEBUG:
    print("[Data Prep] DEBUG MODE: Truncating DataFrames for rapid iteration.")
    train_df = train_df.head(100)
    val_df = val_df.head(100)
    test_df = test_df.head(100)


## 2.6 Vocabulary Construction

The vocabulary is built from training captions only. This avoids validation/test leakage and ensures that out-of-split words are treated realistically as unknown tokens.


In [ ]:
class Vocabulary:
    """
    Handles the bidirectional mapping between string tokens and integer IDs.
    """
    def __init__(self, freq_threshold: int = CFG.VOCAB_MIN_FREQ) -> None:
        self.freq_threshold = freq_threshold

        # Initialize dictionaries with reserved MLOps special tokens
        self.itos: Dict[int, str] = {0: "<PAD>", 1: "<START>", 2: "<END>", 3: "<UNK>"}
        self.stoi: Dict[str, int] = {"<PAD>": 0, "<START>": 1, "<END>": 2, "<UNK>": 3}
        self.idx: int = 4

    def __len__(self) -> int:
        """Returns the total number of unique words in the vocabulary."""
        return len(self.itos)

    def tokenize(self, text: str) -> List[str]:
        """Converts raw text to lowercase and isolates punctuation."""
        text = str(text).lower()
        # Ensure punctuation is treated as separate tokens to improve context learning
        for punc in [".", ",", "!", "?", "\"", "'", "(", ")"]:
            text = text.replace(punc, f" {punc} ")
        return text.split()

    def build_vocabulary(self, sentence_list: List[str]) -> None:
        """Populates the stoi and itos dictionaries based on token frequencies."""
        print(f"--- [Vocabulary] Building index with threshold >= {self.freq_threshold} ---")
        frequencies = Counter()

        for sentence in sentence_list:
            for word in self.tokenize(sentence):
                frequencies[word] += 1

        # Only add words that meet the frequency requirement to prevent overfitting on typos
        for word, count in frequencies.items():
            if count >= self.freq_threshold:
                self.stoi[word] = self.idx
                self.itos[self.idx] = word
                self.idx += 1

        print(f"[Vocabulary] Processed {len(sentence_list)} sentences. Indexed {len(self.itos)} unique tokens.")

    def numericalize(self, text: str) -> List[int]:
        """Converts a raw string into a list of integer tensor indices."""
        tokenized_text = self.tokenize(text)
        return [
            self.stoi.get(token, self.stoi["<UNK>"])
            for token in tokenized_text
        ]

In [ ]:
# 1. Extract all clean training captions
# We ONLY build the vocabulary on the training set to prevent data leakage from the val set
train_captions = train_df['caption'].tolist()

# 2. Instantiate and build the Vocabulary object
vocab = Vocabulary(freq_threshold=CFG.VOCAB_MIN_FREQ)
vocab.build_vocabulary(train_captions)

# 3. Validation Check
if CFG.DEBUG or CFG.MODE == "train":
    # Test the numericalization pipeline on the first available caption
    sample_text = train_captions[0]
    print(f"\n[Validation] Source Text : {sample_text}")
    print(f"[Validation] Tokenized   : {vocab.tokenize(sample_text)}")
    print(f"[Validation] Encoded IDs : {vocab.numericalize(sample_text)}")

In [ ]:
def analyze_nlp_distributions(captions: List[str], vocab: Vocabulary) -> None:
    """
    Analyzes caption lengths and frequent words to inform sequence length and vocabulary choices.
    """
    print("--- [EDA] NLP Token Distribution Analysis ---")

    seq_lengths = [len(vocab.tokenize(cap)) + 2 for cap in captions]

    all_words = []
    for cap in captions:
        all_words.extend(vocab.tokenize(cap))
    word_counts = pd.Series(all_words).value_counts()

    percentile_95 = np.percentile(seq_lengths, 95)

    if CFG.SHOW_EDA_PLOTS:
        fig, axes = plt.subplots(1, 2, figsize=(10, 3.8), dpi=100)
        cmap = plt.colormaps['viridis']

        axes[0].hist(seq_lengths, bins=40, color=cmap(0.55), edgecolor='white')
        axes[0].axvline(percentile_95, color=cmap(0.95), linestyle='--', linewidth=1.5)
        axes[0].set_title('Caption Length Distribution', fontsize=10)
        axes[0].set_xlabel('Tokens including <START>/<END>')
        axes[0].set_ylabel('Count')
        axes[0].grid(axis='y', alpha=0.25)

        top_20 = word_counts.head(20).sort_values()
        axes[1].barh(top_20.index, top_20.values, color=cmap(np.linspace(0.25, 0.85, len(top_20))))
        axes[1].set_title('Top 20 Most Frequent Words', fontsize=10)
        axes[1].set_xlabel('Frequency')
        axes[1].tick_params(axis='y', labelsize=8)
        axes[1].grid(axis='x', alpha=0.25)

        plt.tight_layout()
        plt.show()
        plt.close(fig)

    print(f"\n[NLP Summary]")
    print(f"Shortest Caption : {min(seq_lengths)} tokens")
    print(f"Longest Caption  : {max(seq_lengths)} tokens")
    print(f"Average Caption  : {int(np.mean(seq_lengths))} tokens")
    print(f"95% Threshold    : {int(percentile_95)} tokens")
    print(f"Actionable Insight: Set CFG.MAX_SEQ_LEN to ~{int(percentile_95)} to capture most data without wasting VRAM.")

# Execute the EDA on the clean training captions
if CFG.SHOW_EDA_PLOTS:
    analyze_nlp_distributions(train_captions, vocab)


### Vocabulary and Sequence-Length Takeaways

* `VOCAB_MIN_FREQ = 2` reduces rare-token noise while keeping strong token coverage.
* `MAX_SEQ_LEN = 25` captures most captions without excessive padding.
* Validation/test unknown words are expected and handled with `<UNK>`.


## 2.7 Dataset and DataLoader Setup

This section converts cleaned image-caption rows into PyTorch datasets and loaders for training, validation, and final test evaluation.


In [ ]:
class VizWizDataset(Dataset):
    """
    Custom PyTorch Dataset for loading images and numericalized captions.
    """
    def __init__(self, df: pd.DataFrame, img_dir: Path, vocab: Vocabulary, transform: transforms.Compose, max_seq_len: int):
        self.df = df
        self.img_dir = img_dir
        self.vocab = vocab
        self.transform = transform
        self.max_seq_len = max_seq_len

    def __len__(self) -> int:
        return len(self.df)

    def __getitem__(self, index: int) -> Tuple[torch.Tensor, torch.Tensor]:
        caption = self.df.iloc[index]['caption']
        img_filename = self.df.iloc[index]['file_name']
        img_path = self.img_dir / img_filename

        # 1. Load and Transform Image
        # .convert("RGB") is crucial to drop Alpha channels from PNGs or convert Grayscale
        img = Image.open(img_path).convert("RGB")
        if self.transform is not None:
            img = self.transform(img)

        # 2. Numericalize Text
        numericalized_caption = [self.vocab.stoi["<START>"]]
        numericalized_caption.extend(self.vocab.numericalize(caption))
        numericalized_caption.append(self.vocab.stoi["<END>"])

        # 3. Truncate Extreme Outliers
        # If the caption exceeds our EDA-derived MAX_SEQ_LEN, we cut it short but preserve the <END> token
        if len(numericalized_caption) > self.max_seq_len:
            numericalized_caption = numericalized_caption[:self.max_seq_len - 1]
            numericalized_caption.append(self.vocab.stoi["<END>"])

        return img, torch.tensor(numericalized_caption)


class CapsCollate:
    """
    Custom collate function to handle variable-length text sequences within a batch.
    """
    def __init__(self, pad_idx: int):
        self.pad_idx = pad_idx

    def __call__(self, batch: List[Tuple[torch.Tensor, torch.Tensor]]) -> Tuple[torch.Tensor, torch.Tensor]:
        # Extract images and stack them into a 4D tensor: [batch_size, channels, height, width]
        imgs = [item[0].unsqueeze(0) for item in batch]
        imgs = torch.cat(imgs, dim=0)

        # Extract textual targets
        targets = [item[1] for item in batch]

        # Dynamically pad sequences to the longest sequence *in this specific batch*
        targets = pad_sequence(targets, batch_first=True, padding_value=self.pad_idx)

        return imgs, targets

In [ ]:
print("--- [Data Loader] Initializing PyTorch Pipelines ---")

# 1. Define Standard ImageNet Transformations
transform_pipeline = transforms.Compose([
    transforms.Resize((CFG.IMAGE_SIZE, CFG.IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# 2. Get the Padding Index from the Vocabulary
pad_idx = vocab.stoi["<PAD>"]

# 3. Instantiate Datasets
train_dataset = VizWizDataset(
    df=train_df, img_dir=CFG.TRAIN_IMG_DIR, vocab=vocab,
    transform=transform_pipeline, max_seq_len=CFG.MAX_SEQ_LEN
)

val_dataset = VizWizDataset(
    df=val_df, img_dir=CFG.VAL_IMG_DIR, vocab=vocab,
    transform=transform_pipeline, max_seq_len=CFG.MAX_SEQ_LEN
)

test_dataset = VizWizDataset(
    df=test_df, img_dir=CFG.VAL_IMG_DIR, vocab=vocab,
    transform=transform_pipeline, max_seq_len=CFG.MAX_SEQ_LEN
)

# 4. Instantiate DataLoaders
# pin_memory=True speeds up the transfer of data from CPU RAM to GPU VRAM
train_loader = DataLoader(
    dataset=train_dataset,
    batch_size=CFG.BATCH_SIZE,
    shuffle=True,           # Shuffle training data to break correlation between consecutive batches
    num_workers=CFG.NUM_WORKERS,
    pin_memory=True,
    collate_fn=CapsCollate(pad_idx=pad_idx)
)

val_loader = DataLoader(
    dataset=val_dataset,
    batch_size=CFG.BATCH_SIZE,
    shuffle=False,          # Never shuffle validation data; order doesn't matter for evaluation
    num_workers=CFG.NUM_WORKERS,
    pin_memory=True,
    collate_fn=CapsCollate(pad_idx=pad_idx)
)

test_loader = DataLoader(
    dataset=test_dataset,
    batch_size=CFG.BATCH_SIZE,
    shuffle=False,
    num_workers=CFG.NUM_WORKERS,
    pin_memory=True,
    collate_fn=CapsCollate(pad_idx=pad_idx)
)

print(f"Train Loader : {len(train_loader)} batches (Batch Size: {CFG.BATCH_SIZE})")
print(f"Val Loader   : {len(val_loader)} batches (Batch Size: {CFG.BATCH_SIZE})")
print(f"Test Loader  : {len(test_loader)} batches (Batch Size: {CFG.BATCH_SIZE})")

# 5. Pipeline Sanity Check (Inspect the first batch)
if CFG.DEBUG or CFG.MODE == "train":
    print("\n--- [Data Loader] First Batch Sanity Check ---")
    dataiter = iter(train_loader)
    images, captions = next(dataiter)

    print(f"Images Tensor Shape  : {images.shape}  -> [Batch, Channels, Height, Width]")
    print(f"Captions Tensor Shape: {captions.shape}   -> [Batch, Sequence_Length]")

# 3. Model 1 - Baseline ResNet-LSTM

Model 1 uses a frozen pretrained ResNet-50 encoder and an LSTM decoder. It establishes a clear baseline before adding attention.


## 3.1 Encoder: Global CNN Features

The encoder extracts a single global image representation using a pretrained ResNet-50 backbone. The backbone is frozen to reduce training cost and overfitting risk.


In [ ]:
class EncoderCNN(nn.Module):
    """
    Utilizes a pre-trained ResNet-50 to extract visual features and
    projects them into a shared embedding space.
    """
    def __init__(self, embed_size: int):
        super(EncoderCNN, self).__init__()
        # Load pre-trained ResNet-50
        resnet = models.resnet50(pretrained=True)

        # Freeze parameters so they are not updated during training
        for param in resnet.parameters():
            param.requires_grad = False

        # Remove the final classification layer (fc)
        modules = list(resnet.children())[:-1]
        self.resnet = nn.Sequential(*modules)

        # Linear layer to project features to embed_size
        self.linear = nn.Linear(resnet.fc.in_features, embed_size)
        self.batch_norm = nn.BatchNorm1d(embed_size, momentum=0.01)

    def forward(self, images: torch.Tensor) -> torch.Tensor:
        # Extract features: [batch_size, 2048, 1, 1]
        with torch.no_grad():
            features = self.resnet(images)

        # Flatten: [batch_size, 2048]
        features = features.view(features.size(0), -1)

        # Project and Normalize: [batch_size, embed_size]
        return self.batch_norm(self.linear(features))

## 3.2 Decoder: LSTM Language Model

The decoder receives the projected image feature and learns to generate captions token by token with teacher forcing.


In [ ]:
class DecoderRNN(nn.Module):
    """
    An LSTM-based generator that produces captions conditioned
    on visual features extracted by the Encoder.
    """
    def __init__(self, embed_size: int, hidden_size: int, vocab_size: int, num_layers: int):
        super(DecoderRNN, self).__init__()
        self.embed = nn.Embedding(vocab_size, embed_size)
        self.lstm = nn.LSTM(embed_size, hidden_size, num_layers, batch_first=True)
        self.linear = nn.Linear(hidden_size, vocab_size)

    def forward(self, features: torch.Tensor, captions: torch.Tensor) -> torch.Tensor:
        # Remove the <END> token for training (Teacher Forcing)
        # captions input: [<START>, w1, w2, ..., wN-1]
        embeddings = self.embed(captions[:, :-1])

        # Concatenate image features as the very first 'word' in the sequence
        # combined: [batch_size, sequence_length, embed_size]
        combined = torch.cat((features.unsqueeze(1), embeddings), dim=1)

        # LSTM pass
        hiddens, _ = self.lstm(combined)

        # Project to vocabulary logits
        outputs = self.linear(hiddens)

        # Discard the very first prediction (which was based on the image feature alone)
        # to align with targets: [w1, w2, ..., wN-1, <END>]
        return outputs[:, 1:, :]

## 3.3 Baseline Model Assembly

The wrapper connects the CNN encoder and LSTM decoder into one trainable image-captioning model.


In [ ]:
class ImageCaptioningModel(nn.Module):
    """
    Unified interface for the VisionVoice architecture.
    Coordinates the flow between visual extraction and text generation.
    """
    def __init__(self, encoder: nn.Module, decoder: nn.Module):
        super(ImageCaptioningModel, self).__init__()
        self.encoder = encoder
        self.decoder = decoder

    def forward(self, images: torch.Tensor, captions: torch.Tensor) -> torch.Tensor:
        # Step 1: Extract visual features
        features = self.encoder(images)
        # Step 2: Generate sequence predictions
        outputs = self.decoder(features, captions)
        return outputs

# --- Model Instantiation ---

# Initialize the sub-modules using our centralized CFG parameters
encoder_net = EncoderCNN(embed_size=CFG.EMBED_SIZE)
decoder_net = DecoderRNN(
    embed_size=CFG.EMBED_SIZE,
    hidden_size=CFG.HIDDEN_SIZE,
    vocab_size=len(vocab),
    num_layers=CFG.NUM_LAYERS
)

# Combine into the final model and transfer to the target DEVICE (CUDA/MPS/CPU)
cap_model = ImageCaptioningModel(encoder_net, decoder_net).to(DEVICE)

# --- Parameter Audit ---
if CFG.MODE == "train" or CFG.DEBUG:
    total_params = sum(p.numel() for p in cap_model.parameters())
    trainable_params = sum(p.numel() for p in cap_model.parameters() if p.requires_grad)

    print(f"--- [Architecture Audit] ---")
    print(f"Total Parameters     : {total_params:,}")
    print(f"Trainable Parameters : {trainable_params:,}")
    print(f"Frozen Parameters    : {total_params - trainable_params:,}")
    print(f"Target Device        : {DEVICE}")

    # Verify that the majority of parameters (the ResNet backbone) are frozen
    freeze_ratio = (1 - (trainable_params / total_params)) * 100
    print(f"Backbone Freeze Ratio: {freeze_ratio:.1f}%")

## 3.4 Loss Function and Optimizer

Cross-entropy loss is used at each time step, with padding ignored so variable-length captions do not distort the objective.


In [ ]:
# 1. Define Loss Function
# ignore_index ensures the model isn't penalized/rewarded for padding tokens
criterion = nn.CrossEntropyLoss(ignore_index=vocab.stoi["<PAD>"])

# 2. Define Optimizer
# We only pass the trainable parameters (Projection + LSTM + Linear layers)
optimizer = optim.Adam(
    filter(lambda p: p.requires_grad, cap_model.parameters()),
    lr=CFG.LEARNING_RATE
)

print(f"--- [Optimization] ---")
print(f"Loss Function : {criterion}")
print(f"Optimizer     : Adam (LR={CFG.LEARNING_RATE})")

## 3.5 Baseline Training

The baseline model is trained with validation monitoring and early-stopping logic. The best checkpoint is selected by validation loss.


In [ ]:
def train_with_early_stopping(
    model: nn.Module,
    train_loader: DataLoader,
    val_loader: DataLoader,
    criterion: nn.Module,
    optimizer: optim.Optimizer,
    device: torch.device,
    epochs: int,
    patience: int = 3
) -> Tuple[List[float], List[float]]:
    """
    Trains the model with a validation pass and Early Stopping logic.
    Saves only the best model weights based on Validation Loss.
    """
    train_losses = []
    val_losses = []

    best_val_loss = float('inf')
    epochs_without_improvement = 0
    best_model_path = CFG.WORK_DIR / "vision_voice_baseline_best.pth"

    print(f"--- [Training] Starting Baseline with Early Stopping (Patience: {patience}) ---")

    for epoch in range(epochs):
        # --- 1. Training Phase ---
        model.train()
        running_train_loss = 0.0
        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs} [Train]", leave=False)

        for images, captions in pbar:
            images, captions = images.to(device), captions.to(device)
            optimizer.zero_grad()
            outputs = model(images, captions)
            targets = captions[:, 1:]
            loss = criterion(outputs.reshape(-1, outputs.shape[2]), targets.reshape(-1))
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=CFG.MAX_GRAD_NORM)
            optimizer.step()
            running_train_loss += loss.item()

        avg_train_loss = running_train_loss / len(train_loader)
        train_losses.append(avg_train_loss)

        # --- 2. Validation Phase ---
        model.eval()
        running_val_loss = 0.0
        with torch.no_grad():
            for images, captions in val_loader:
                images, captions = images.to(device), captions.to(device)
                outputs = model(images, captions)
                targets = captions[:, 1:]
                loss = criterion(outputs.reshape(-1, outputs.shape[2]), targets.reshape(-1))
                running_val_loss += loss.item()

        avg_val_loss = running_val_loss / len(val_loader)
        val_losses.append(avg_val_loss)

        print(f"Epoch {epoch+1} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")

        # --- 3. Early Stopping & Best Model Saving ---
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            epochs_without_improvement = 0
            # Save the best model only
            torch.save(model.state_dict(), best_model_path)
            print(f"   --> New Best Val Loss! Model saved to {best_model_path.name}")
        else:
            epochs_without_improvement += 1
            print(f"   --> No improvement for {epochs_without_improvement} epoch(s).")

        if epochs_without_improvement >= patience:
            print(f"\n[Early Stopping] Validation loss hasn't improved in {patience} epochs. Hitting the brakes.")
            break

    return train_losses, val_losses

# --- Execute Training ---
if CFG.MODE == "train":
    clear_memory()
    train_log, val_log = train_with_early_stopping(
        model=cap_model,
        train_loader=train_loader,
        val_loader=val_loader,
        criterion=criterion,
        optimizer=optimizer,
        device=DEVICE,
        epochs=CFG.EPOCHS,
        patience=3 # Stop if 3 epochs pass with no improvement
    )

## 3.6 Baseline Loss Curve

The loss curve helps check whether the model is learning, overfitting, or underfitting.


In [ ]:
if CFG.SHOW_MODEL_PLOTS:
    fig, ax = plt.subplots(figsize=(7, 4), dpi=100)
    cmap = plt.colormaps['viridis']
    ax.plot(train_log, label='Train Loss', color=cmap(0.35), linewidth=2)
    ax.plot(val_log, label='Val Loss', color=cmap(0.85), linewidth=2)
    ax.set_title('VisionVoice Baseline Training Loss', fontsize=11)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Cross-Entropy Loss')
    ax.grid(alpha=0.25)
    ax.legend(frameon=False)
    plt.tight_layout()
    plt.show()
    plt.close(fig)


# 4. Model 1 Evaluation


## 4.1 Baseline Inference

At inference time the decoder no longer receives ground-truth tokens. It generates captions autoregressively using greedy decoding.


In [ ]:
def generate_caption(model: nn.Module, image: torch.Tensor, vocab: Vocabulary) -> List[str]:
    """
    Generates a caption for a single image using greedy search.
    """
    model.eval()
    result_caption = []

    with torch.no_grad():
        # Add batch dimension and move to device
        image = image.unsqueeze(0).to(DEVICE)
        
        # 1. Get visual features from Encoder
        features = model.encoder(image)
        states = None # Initialize LSTM hidden/cell states
        
        # 2. Start with the <START> token
        current_word_idx = torch.tensor([vocab.stoi["<START>"]]).to(DEVICE)
        
        for _ in range(CFG.MAX_SEQ_LEN):
            # 3. Predict the next word
            embeddings = model.decoder.embed(current_word_idx).unsqueeze(0)
            hiddens, states = model.decoder.lstm(embeddings, states)
            outputs = model.decoder.linear(hiddens.squeeze(0))
            
            # 4. Greedy Search: Pick the word with the highest probability
            predicted_idx = outputs.argmax(1)
            predicted_word = vocab.itos[predicted_idx.item()]
            
            if predicted_word == "<END>":
                break
                
            result_caption.append(predicted_word)
            current_word_idx = predicted_idx

    return result_caption

## 4.2 Baseline BLEU Evaluation

BLEU-1 through BLEU-4 are computed on the internal test split. This provides a quantitative lower bound for the refined model.


In [ ]:
def calculate_corpus_bleu(model: nn.Module, df: pd.DataFrame, vocab: Vocabulary):
    """
    Evaluates the model across the test set and calculates BLEU scores.
    """
    print(f"--- [Evaluation] Calculating test BLEU scores on {CFG.BLEU_EVAL_SAMPLES} samples ---")
    
    # Load the best weights saved during training
    best_path = CFG.WORK_DIR / "vision_voice_baseline_best.pth"
    model.load_state_dict(torch.load(best_path, map_location=DEVICE))
    model.to(DEVICE)
    model.eval()

    references_corpus = []
    hypotheses_corpus = []
    
    # Group by image to get all available clean references
    grouped = df.groupby('file_name')
    sample_files = random.sample(list(grouped.groups.keys()), min(CFG.BLEU_EVAL_SAMPLES, len(grouped)))

    for file_name in tqdm(sample_files, desc="Generating Captions"):
        img_path = CFG.VAL_IMG_DIR / file_name
        
        # 1. Prepare References
        group = grouped.get_group(file_name)
        refs = [vocab.tokenize(row['caption']) for _, row in group.iterrows()]
        references_corpus.append(refs)
        
        # 2. Generate Hypothesis
        img = Image.open(img_path).convert("RGB")
        img_tensor = transform_pipeline(img)
        hypothesis = generate_caption(model, img_tensor, vocab)
        hypotheses_corpus.append(hypothesis)

    # 3. Calculate Scores
    b1 = corpus_bleu(references_corpus, hypotheses_corpus, weights=(1.0, 0, 0, 0))
    b2 = corpus_bleu(references_corpus, hypotheses_corpus, weights=(0.5, 0.5, 0, 0))
    b3 = corpus_bleu(references_corpus, hypotheses_corpus, weights=(0.33, 0.33, 0.33, 0))
    b4 = corpus_bleu(references_corpus, hypotheses_corpus, weights=(0.25, 0.25, 0.25, 0.25))

    print(f"\n[Final Results]")
    print(f"BLEU-1: {b1:.4f} | BLEU-2: {b2:.4f} | BLEU-3: {b3:.4f} | BLEU-4: {b4:.4f}")
    return b1, b2, b3, b4

# Run evaluation
if CFG.MODE in {"train", "inference"}:
    b_results = calculate_corpus_bleu(cap_model, test_df, vocab)

## 4.3 Baseline Qualitative Inspection

Visual inspection checks whether generated captions are actually grounded in the image, not merely fluent.


In [ ]:
def inspect_model_visually(model, df, vocab, num_samples=3):
    """
    Displays random validation images with predicted vs. reference captions.
    """
    model.eval()
    
    # Group validation data to access all references
    grouped = df.groupby('file_name')
    sample_files = random.sample(list(grouped.groups.keys()), num_samples)

    for file_name in sample_files:
        img_path = CFG.VAL_IMG_DIR / file_name
        
        # 1. Generate Prediction
        img = Image.open(img_path).convert("RGB")
        img_tensor = transform_pipeline(img)
        predicted_words = generate_caption(model, img_tensor, vocab)
        predicted_sentence = " ".join(predicted_words).capitalize() + "."

        # 2. Setup Visualization
        plt.figure(figsize=(5, 5), dpi=100)
        plt.imshow(img)
        plt.axis('off')
        plt.show()

        print(f"--- [Qualitative Check] {file_name} ---")
        # Bold and Color formatting for professional output
        print(f"\033[1mPredicted: \033[92m{predicted_sentence}\033[0m") 
        
        print("\nGround Truth References:")
        group = grouped.get_group(file_name)
        for i, (_, row) in enumerate(group.iterrows(), 1):
            print(f"  {i}. {row['caption']}")
        print("="*60 + "\n")

# Execute visual check
if CFG.SHOW_MODEL_PLOTS:
    inspect_model_visually(cap_model, test_df, vocab, num_samples=3)

# 5. Phase 3 Rationale

Phase 2 showed that the baseline ResNet-LSTM can learn the broad caption distribution, but qualitative inspection exposed a serious failure mode: repeated generic captions that are fluent but weakly grounded in the image. Based on this insight, Model 2 keeps the ResNet/LSTM foundation but changes the visual interface from a single global feature vector to a spatial feature map with Bahdanau-style attention.


# 6. Model 2 - ResNet-LSTM with Attention

The refined architecture adds spatial attention so each generated word can condition on different regions of the image.


## 6.1 Encoder: Spatial Feature Map

Instead of collapsing the image into one global vector, the encoder keeps a 7x7 grid of ResNet features. This gives the decoder spatial evidence to attend over.


In [ ]:
class EncoderAttention(nn.Module):
    def __init__(self, encoded_image_size=7):
        super(EncoderAttention, self).__init__()
        resnet = models.resnet50(pretrained=True)
        
        # Remove the last two layers (Pooling and FC)
        modules = list(resnet.children())[:-2]
        self.resnet = nn.Sequential(*modules)
        
        # Resize image to a fixed spatial size (7x7)
        self.adaptive_pool = nn.AdaptiveAvgPool2d((encoded_image_size, encoded_image_size))
        
        # Freeze early layers, but we will unfreeze the last block for fine-tuning
        for param in self.resnet.parameters():
            param.requires_grad = False
        for child in list(self.resnet.children())[5:]:
            for param in child.parameters():
                param.requires_grad = True

    def forward(self, images):
        # Out: [batch, 2048, 7, 7]
        out = self.resnet(images)
        out = self.adaptive_pool(out)
        # Reshape to: [batch, 49, 2048]
        out = out.permute(0, 2, 3, 1).view(out.size(0), -1, 2048)
        return out

## 6.2 Attention Mechanism

The attention module scores each spatial feature against the current decoder hidden state and produces a context vector for the next word.


In [ ]:
class Attention(nn.Module):
    def __init__(self, encoder_dim, decoder_dim, attention_dim):
        super(Attention, self).__init__()
        self.encoder_att = nn.Linear(encoder_dim, attention_dim)
        self.decoder_att = nn.Linear(decoder_dim, attention_dim)
        self.full_att = nn.Linear(attention_dim, 1)
        self.relu = nn.ReLU()
        self.softmax = nn.Softmax(dim=1)

    def forward(self, encoder_out, decoder_hidden):
        # encoder_out: [batch, 49, 2048] | decoder_hidden: [batch, 512]
        att1 = self.encoder_att(encoder_out)        # [batch, 49, attention_dim]
        att2 = self.decoder_att(decoder_hidden)     # [batch, attention_dim]
        
        # Broadcast hidden state across all 49 locations and combine
        att = self.full_att(self.relu(att1 + att2.unsqueeze(1))).squeeze(2) 
        
        # Alpha (weights) sum to 1.0
        alpha = self.softmax(att) 
        
        # Calculate context vector: weighted sum of image features
        context = (encoder_out * alpha.unsqueeze(2)).sum(dim=1) 
        
        return context, alpha

In [ ]:
class DecoderWithAttention(nn.Module):
    def __init__(self, vocab_size, embed_size, decoder_dim, encoder_dim=2048, attention_dim=512):
        super(DecoderWithAttention, self).__init__()
        self.attention = Attention(encoder_dim, decoder_dim, attention_dim)
        self.embedding = nn.Embedding(vocab_size, embed_size)
        self.dropout = nn.Dropout(p=0.5)
        
        # Input to LSTM is word embedding + context vector
        self.decode_step = nn.LSTMCell(embed_size + encoder_dim, decoder_dim, bias=True)
        self.fc = nn.Linear(decoder_dim, vocab_size)
        
        # Linear layers to initialize LSTM states from image average
        self.init_h = nn.Linear(encoder_dim, decoder_dim)
        self.init_c = nn.Linear(encoder_dim, decoder_dim)

    def forward(self, encoder_out, captions):
        batch_size = encoder_out.size(0)
        vocab_size = self.fc.out_features
        
        # 1. Initialize LSTM states based on global image average
        mean_encoder_out = encoder_out.mean(dim=1)
        h, c = self.init_h(mean_encoder_out), self.init_c(mean_encoder_out)
        
        embeddings = self.embedding(captions)
        
        # We need to loop manually because attention changes at every step
        seq_len = captions.size(1) - 1
        predictions = torch.zeros(batch_size, seq_len, vocab_size).to(DEVICE)
        
        for t in range(seq_len):
            context, _ = self.attention(encoder_out, h)
            lstm_input = torch.cat([embeddings[:, t, :], context], dim=1)
            h, c = self.decode_step(lstm_input, (h, c))
            preds = self.fc(self.dropout(h))
            predictions[:, t, :] = preds
            
        return predictions

## 6.3 Attention Model Assembly

The full attention model connects the spatial encoder and attention decoder into a single trainable architecture.


In [ ]:
class VisionVoiceAttention(nn.Module):
    """
    Model 2: Encoder-Decoder with Bahdanau Attention.
    Designed to fix 'template repetition' by grounding each word 
    in specific spatial regions of the image.
    """
    def __init__(self, encoder, decoder):
        super(VisionVoiceAttention, self).__init__()
        self.encoder = encoder
        self.decoder = decoder

    def forward(self, images, captions):
        # 1. Extract spatial feature maps: [batch, 49, 2048]
        encoder_out = self.encoder(images)
        
        # 2. Decode with attention: [batch, seq_len, vocab_size]
        outputs = self.decoder(encoder_out, captions)
        return outputs

# --- Model 2 Instantiation ---

# Parameters for our refined architecture
ATTENTION_DIM = 512
DECODER_DIM = 512  # Increased for better linguistic capacity
EMBED_SIZE = 256

refined_encoder = EncoderAttention()
refined_decoder = DecoderWithAttention(
    vocab_size=len(vocab),
    embed_size=EMBED_SIZE,
    decoder_dim=DECODER_DIM,
    attention_dim=ATTENTION_DIM
)

# Move to GPU/MPS
model_v2 = VisionVoiceAttention(refined_encoder, refined_decoder).to(DEVICE)

print(f"--- [Model 2 Assembly] ---")
print(f"Architecture: ResNet-50 + Bahdanau Attention + LSTM")
print(f"Device      : {DEVICE}")

## 6.4 Optimizer Strategy

The decoder is trained with a larger learning rate, while unfrozen encoder layers use a smaller fine-tuning rate to protect pretrained visual features.


In [ ]:
# Define separate parameter groups with different learning rates
optimizer_v2 = optim.Adam([
    {'params': model_v2.decoder.parameters(), 'lr': 1e-4},
    # Only fine-tune the encoder layers that are NOT frozen
    {'params': filter(lambda p: p.requires_grad, model_v2.encoder.parameters()), 'lr': 1e-5}
])

# We use the same CrossEntropyLoss (ignoring PAD)
criterion_v2 = nn.CrossEntropyLoss(ignore_index=vocab.stoi["<PAD>"])

print(f"--- [Model 2 Optimization] ---")
print(f"Decoder LR: 1e-4 | Encoder (Fine-tune) LR: 1e-5")

## 6.5 Attention Training

The refined model is trained with validation monitoring and checkpointing, matching the baseline evaluation protocol.


In [ ]:
def train_attention_model(
    model, train_loader, val_loader, criterion, optimizer, device, epochs, patience=3
):
    train_losses, val_losses = [], []
    best_val_loss = float('inf')
    epochs_without_improvement = 0
    best_path = CFG.WORK_DIR / "vision_voice_attention_best.pth"

    print(f"--- [Training Phase 3] Fine-tuning with Attention ---")

    for epoch in range(epochs):
        # --- Training ---
        model.train()
        running_train_loss = 0.0
        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs} [Train]", leave=False)
        
        for images, captions in pbar:
            images, captions = images.to(device), captions.to(device)
            
            optimizer.zero_grad()
            
            # Forward Pass
            # Model 2 expects (images, captions) and handles the step-by-step logic
            outputs = model(images, captions)
            
            # Targets: Shifted right (ignore <START>)
            targets = captions[:, 1:]
            
            # Flatten for CrossEntropy
            loss = criterion(outputs.reshape(-1, outputs.shape[2]), targets.reshape(-1))
            
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=CFG.MAX_GRAD_NORM)
            optimizer.step()
            
            running_train_loss += loss.item()

        avg_train_loss = running_train_loss / len(train_loader)
        train_losses.append(avg_train_loss)

        # --- Validation ---
        model.eval()
        running_val_loss = 0.0
        with torch.no_grad():
            for images, captions in val_loader:
                images, captions = images.to(device), captions.to(device)
                outputs = model(images, captions)
                targets = captions[:, 1:]
                loss = criterion(outputs.reshape(-1, outputs.shape[2]), targets.reshape(-1))
                running_val_loss += loss.item()
        
        avg_val_loss = running_val_loss / len(val_loader)
        val_losses.append(avg_val_loss)

        print(f"Epoch {epoch+1} | Train: {avg_train_loss:.4f} | Val: {avg_val_loss:.4f}")

        # --- Best Model & Early Stopping ---
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            epochs_without_improvement = 0
            torch.save(model.state_dict(), best_path)
            print(f"   --> Attention Model Saved: {best_path.name}")
        else:
            epochs_without_improvement += 1
            if epochs_without_improvement >= patience:
                print(f"[Early Stopping] Training halted at epoch {epoch+1}")
                break

    return train_losses, val_losses

# --- Execution ---
if CFG.MODE == "train":
    clear_memory()
    attention_train_log, attention_val_log = train_attention_model(
        model=model_v2,
        train_loader=train_loader,
        val_loader=val_loader,
        criterion=criterion_v2,
        optimizer=optimizer_v2,
        device=DEVICE,
        epochs=CFG.EPOCHS
    )

# 7. Model 2 Evaluation


## 7.1 Attention Inference

The default decoder uses greedy search for comparability with the baseline. Optional beam search is included for further experiments.


In [ ]:
def generate_caption_with_attention(model, image, vocab):
    """
    Generates a caption with greedy decoding and returns attention weights for visualization.
    """
    model.eval()
    result_caption = []
    alphas = []

    with torch.no_grad():
        image = image.unsqueeze(0).to(DEVICE)
        encoder_out = model.encoder(image)
        mean_encoder_out = encoder_out.mean(dim=1)
        h = model.decoder.init_h(mean_encoder_out)
        c = model.decoder.init_c(mean_encoder_out)
        word = torch.tensor([vocab.stoi["<START>"]]).to(DEVICE)

        for _ in range(CFG.MAX_SEQ_LEN):
            context, alpha = model.decoder.attention(encoder_out, h)
            alphas.append(alpha.cpu().numpy())

            embeddings = model.decoder.embedding(word)
            lstm_input = torch.cat([embeddings, context], dim=1)
            h, c = model.decoder.decode_step(lstm_input, (h, c))

            output = model.decoder.fc(h)
            predicted_idx = output.argmax(1)
            predicted_word = vocab.itos[predicted_idx.item()]

            if predicted_word == "<END>":
                break

            result_caption.append(predicted_word)
            word = predicted_idx

    return result_caption, alphas


def generate_caption_with_attention_beam(model, image, vocab, beam_size: int = CFG.BEAM_SIZE):
    """
    Optional beam-search decoder for quantitative experiments.
    Greedy remains the default so previous results are reproducible.
    """
    model.eval()
    start_idx = vocab.stoi["<START>"]
    end_idx = vocab.stoi["<END>"]
    blocked = {vocab.stoi["<PAD>"], vocab.stoi["<START>"], vocab.stoi["<UNK>"]}

    with torch.no_grad():
        image = image.unsqueeze(0).to(DEVICE)
        encoder_out = model.encoder(image)
        mean_encoder_out = encoder_out.mean(dim=1)
        h0 = model.decoder.init_h(mean_encoder_out)
        c0 = model.decoder.init_c(mean_encoder_out)

        beams = [([start_idx], 0.0, h0, c0)]
        completed = []

        for _ in range(CFG.MAX_SEQ_LEN):
            candidates = []
            for token_ids, score, h, c in beams:
                last_idx = token_ids[-1]
                if last_idx == end_idx:
                    completed.append((token_ids, score))
                    continue

                word = torch.tensor([last_idx], device=DEVICE)
                context, _ = model.decoder.attention(encoder_out, h)
                embedding = model.decoder.embedding(word)
                lstm_input = torch.cat([embedding, context], dim=1)
                next_h, next_c = model.decoder.decode_step(lstm_input, (h, c))
                logits = model.decoder.fc(next_h).squeeze(0)

                for blocked_idx in blocked:
                    logits[blocked_idx] = -float('inf')

                log_probs = torch.log_softmax(logits, dim=0)
                top_scores, top_indices = torch.topk(log_probs, beam_size)

                for log_prob, next_idx in zip(top_scores.tolist(), top_indices.tolist()):
                    repeat_penalty = CFG.REPETITION_PENALTY if next_idx in token_ids else 1.0
                    candidates.append((
                        token_ids + [next_idx],
                        score + (log_prob / repeat_penalty),
                        next_h,
                        next_c,
                    ))

            if not candidates:
                break

            beams = sorted(
                candidates,
                key=lambda item: item[1] / (len(item[0]) ** 0.7),
                reverse=True
            )[:beam_size]

        completed.extend((tokens, score) for tokens, score, _, _ in beams)
        best_tokens, _ = max(completed, key=lambda item: item[1] / (len(item[0]) ** 0.7))

    words = []
    for idx in best_tokens[1:]:
        if idx == end_idx:
            break
        words.append(vocab.itos[idx])
    return words


## 7.2 Attention BLEU Evaluation

BLEU-1 through BLEU-4 are computed on the internal test split. Metrics are also saved to JSON for easier reporting.


In [ ]:
def evaluate_attention_model(model, df, vocab, decoding_strategy: str = CFG.DECODING_STRATEGY):
    print(f"--- [Evaluation] Evaluating Model 2 (Attention: {decoding_strategy}) ---")

    best_path = CFG.WORK_DIR / "vision_voice_attention_best.pth"
    model.load_state_dict(torch.load(best_path, map_location=DEVICE))
    model.to(DEVICE)
    model.eval()

    references_corpus = []
    hypotheses_corpus = []

    grouped = df.groupby('file_name')
    sample_files = random.sample(list(grouped.groups.keys()), min(CFG.BLEU_EVAL_SAMPLES, len(grouped)))

    for file_name in tqdm(sample_files, desc="Evaluating"):
        img_path = CFG.VAL_IMG_DIR / file_name

        group = grouped.get_group(file_name)
        refs = [vocab.tokenize(row['caption']) for _, row in group.iterrows()]
        references_corpus.append(refs)

        img = Image.open(img_path).convert("RGB")
        img_tensor = transform_pipeline(img)
        if decoding_strategy == "beam":
            hypothesis = generate_caption_with_attention_beam(model, img_tensor, vocab, beam_size=CFG.BEAM_SIZE)
        else:
            hypothesis, _ = generate_caption_with_attention(model, img_tensor, vocab)
        hypotheses_corpus.append(hypothesis)

    metrics = {
        "model": "attention_resnet_lstm",
        "decoding_strategy": decoding_strategy,
        "eval_samples": len(sample_files),
        "bleu_1": corpus_bleu(references_corpus, hypotheses_corpus, weights=(1.0, 0, 0, 0)),
        "bleu_2": corpus_bleu(references_corpus, hypotheses_corpus, weights=(0.5, 0.5, 0, 0)),
        "bleu_3": corpus_bleu(references_corpus, hypotheses_corpus, weights=(1/3, 1/3, 1/3, 0)),
        "bleu_4": corpus_bleu(references_corpus, hypotheses_corpus, weights=(0.25, 0.25, 0.25, 0.25)),
    }

    print("\n[Model 2 Results]")
    print(
        f"BLEU-1: {metrics['bleu_1']:.4f} | BLEU-2: {metrics['bleu_2']:.4f} | "
        f"BLEU-3: {metrics['bleu_3']:.4f} | BLEU-4: {metrics['bleu_4']:.4f}"
    )

    metrics_path = CFG.WORK_DIR / f"attention_metrics_{decoding_strategy}.json"
    with open(metrics_path, "w", encoding="utf-8") as f:
        json.dump(metrics, f, indent=2)
    print(f"Saved metrics to: {metrics_path}")

    return metrics

# Execute
if CFG.MODE == "train":
    attention_metrics = evaluate_attention_model(model_v2, test_df, vocab)


## 7.3 Attention Visualization

Attention maps show which image regions influenced each generated word. This helps diagnose whether improved captions are visually grounded.


In [ ]:
def visualize_attention(model, df, vocab, num_samples=2):
    """
    Generates a caption and visualizes the attention heatmap for each predicted word.
    """
    model.eval()
    grouped = df.groupby('file_name')
    sample_files = random.sample(list(grouped.groups.keys()), num_samples)

    for file_name in sample_files:
        img_path = CFG.VAL_IMG_DIR / file_name
        img_raw = Image.open(img_path).convert("RGB")
        img_tensor = transform_pipeline(img_raw)
        
        # 1. Generate Caption and Attention Weights
        words, alphas = generate_caption_with_attention(model, img_tensor, vocab)
        
        print(f"\n--- [Model 2 Check] {file_name} ---")
        print(f"\033[1mPredicted: \033[94m{' '.join(words).capitalize()}.\033[0m")
        
        # 2. Plotting the Attention "Focus" per word
        num_words = len(words)
        cols = 4
        rows = (num_words // cols) + 1
        
        fig = plt.figure(figsize=(10, rows * 2.5), dpi=100)
        
        for i, word in enumerate(words):
            # Upscale 7x7 attention map to original image size
            alpha_map = alphas[i].reshape(7, 7)
            alpha_img = Image.fromarray(alpha_map).resize((224, 224), resample=Image.BILINEAR)
            
            ax = fig.add_subplot(rows, cols, i + 1)
            ax.set_title(f"Word: {word}")
            ax.imshow(img_raw.resize((224, 224)))
            # Overlay the heatmap
            ax.imshow(np.array(alpha_img), alpha=0.6, cmap='viridis')
            ax.axis('off')
            
        plt.tight_layout()
        plt.show()
        
        print("Ground Truth References:")
        for i, (_, row) in enumerate(grouped.get_group(file_name).iterrows(), 1):
            print(f"  {i}. {row['caption']}")
        print("="*60)

# Run Visual Inspection
if CFG.SHOW_MODEL_PLOTS:
    visualize_attention(model_v2, test_df, vocab, num_samples=2)